# Case 3: dynamic / DYNOTEARS debugger

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from causal_opt.simulation import simulate_svar
from causal_opt.methods.dynamic_latent import fit_dynamic_latent, lagged_views
from causal_opt.evaluation.dynamic_metrics import dynamic_metrics
cfg=dict(T=300,d=5,p=1,seed=1,s0=5,burn_in=100,lag_sparsity=.25)
data=simulate_svar(**cfg)
X0,Xlags=lagged_views(data.X,cfg['p'])
print(data.X.shape,X0.shape,Xlags.shape,'radius=',data.spectral_radius,'condition=',data.condition_number)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(13,3)); ax[0].plot(data.X[:100,:3]); ax[0].set_title('Observed series'); ax[1].imshow(data.W0_true,cmap='coolwarm'); ax[1].set_title('True W0'); ax[2].imshow(data.W_lags_true[0],cmap='coolwarm'); ax[2].set_title('True W lag 1'); plt.tight_layout()

In [ ]:
ours=fit_dynamic_latent(data.X,p=1,k=0,lambda_0=.1,lambda_lag=.05,wlag_threshold=.2,max_outer_iter=30,inner_max_iter=300)
metrics=dynamic_metrics(data.W0_true,data.W_lags_true,ours.W0,ours.W_lags,data.X,w0_threshold=0,wlag_threshold=0)
print(metrics); print('runtime',ours.runtime_seconds)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(8,3)); ax[0].imshow(ours.W0,cmap='coolwarm'); ax[0].set_title('Estimated W0'); ax[1].imshow(ours.W_lags[0],cmap='coolwarm'); ax[1].set_title('Estimated W lag 1'); plt.tight_layout()
h=ours.diagnostics['history']; fig,ax=plt.subplots(1,4,figsize=(16,3)); ax[0].semilogy([max(r['h'],1e-18) for r in h]); ax[0].set_title('Acyclicity h(W0) across outer iterations'); ax[1].plot([r['fit_loss'] for r in h]); ax[1].set_title('Data-fit loss across augmented-Lagrangian outer iterations'); ax[2].semilogy([r['rho'] for r in h]); ax[2].set_title('rho'); ax[3].plot([r['alpha'] for r in h]); ax[3].set_title('alpha'); plt.tight_layout()

In [ ]:
try:
    from causal_opt.baselines.dynotears import fit_dynotears
    baseline=fit_dynotears(data.X,p=1); print(dynamic_metrics(data.W0_true,data.W_lags_true,baseline.W0,baseline.W_lags,data.X,w0_threshold=0,wlag_threshold=0),baseline.runtime_seconds)
except ImportError as exc: print('DYNOTEARS unavailable:',exc)